# Метод стрельбы: решение обратной баллистической задачи #

Демонстрация метода стрельбы для решения обратной баллистической задачи.
Находим угол стрельбы, при котором снаряд попадает точно в цель.

## Импорты ##

In [ ]:
import numpy as np
from lesson3_utils import plot_trajectories, plot_convergence

## Параметры системы ##

In [ ]:
# Ускорение свободного падения, м/с²
g = 9.81

# Коэффициент сопротивления воздуха, 1/с
k = 0.01

# Скорость горизонтального ветра, м/с
wind_x = 2.0

# Начальная скорость снаряда, м/с
v0 = 50.0

# Целевая точка (x_target, y_target), м
x_target = 200.0
y_target = 0.0

# Начальная точка (x_start, y_start), м
x_start = 0.0
y_start = 0.0

# Точность решения
tolerance = 1e-6

# Максимальное время полета для поиска, с
t_max = 20.0

# Число шагов интегрирования
n_steps = 1000

# Шаг по времени
dt = t_max / n_steps

## 1. Система уравнений движения ##

In [ ]:
def equations_of_motion(state):
    """
    Вычисление правых частей системы дифференциальных уравнений движения снаряда.

    Модель учитывает:
    - Гравитационное ускорение (направлено вниз)
    - Сопротивление воздуха (пропорционально скорости)
    - Горизонтальный ветер

    Args:
        state (numpy.ndarray): вектор состояния [t, x, y, vx, vy], где
                               t - время, y - координаты (метры), vx, vy - скорости (м/с)

    Returns:
        numpy.ndarray: вектор производных [1, dx/dt, dy/dt, dvx/dt, dvy/dt]
                       dx/dt, dy/dt - скорости (м/с)
                       dvx/dt, dvy/dt - ускорения (м/с²)
    """
    t, x, y, vx, vy = state

    # Производные координат (скорости)
    dx_dt = vx  # скорость по x
    dy_dt = vy  # скорость по y

    # Производные скоростей (ускорения)
    # Горизонтальное ускорение: сопротивление воздуха + ветер
    dvx_dt = -k * (vx - wind_x)
    # Вертикальное ускорение: гравитация + сопротивление воздуха
    dvy_dt = -g - k * vy

    return np.array([1, dx_dt, dy_dt, dvx_dt, dvy_dt])

In [ ]:
def rk4_step(state, t):
    """
    Один шаг метода Рунге-Кутты 4-го порядка

    Args:
        state (numpy.ndarray): текущее состояние системы [x, y, vx, vy]
        t (float): текущее время, секунды

    Returns:
        numpy.ndarray: состояние системы на следующем временном шаге [x, y, vx, vy]
    """
    k1 = dt * equations_of_motion(state, t)
    k2 = dt * equations_of_motion(state + 0.5 * k1, t + 0.5 * dt)
    k3 = dt * equations_of_motion(state + 0.5 * k2, t + 0.5 * dt)
    k4 = dt * equations_of_motion(state + k3, t + dt)

    return state + (k1 + 2*k2 + 2*k3 + k4) / 6

## 2. Функции интегрирования траектории ##

In [ ]:
def initialize_state(theta):
    """
    Инициализация вектора состояния снаряда для заданного угла стрельбы.

    Args:
        theta (float): начальный угол стрельбы, радианы

    Returns:
        numpy.ndarray: вектор состояния [x, y, vx, vy] в начальный момент времени
    """
    vx0 = v0 * np.cos(theta)
    vy0 = v0 * np.sin(theta)
    return np.array([x_start, y_start, vx0, vy0])

In [ ]:
def integrate_until_ground(initial_state):
    """
    Интегрирование траектории снаряда до момента падения на землю (y <= 0).

    Args:
        initial_state (numpy.ndarray): начальный вектор состояния [x, y, vx, vy]
        return_trajectory (bool): если True, возвращает всю траекторию

    Returns:
        tuple: (финальное состояние, траектория если запрошена)
               финальное состояние: numpy.ndarray [x, y, vx, vy]
               траектория: list[numpy.ndarray]
    """
    state = initial_state.copy()
    trajectory = [state.copy()]

    t = 0.0

    # Интегрируем до тех пор, пока не достигнем земли или не превысим максимальное время
    while t < t_max and state[1] >= 0:
        state = rk4_step(state, t)
        t += dt

        trajectory.append(state.copy())

    return state, trajectory

In [ ]:
def interpolate_ground_impact(trajectory):
    """
    Интерполяция траектории для точного определения точки падения (y = 0).

    Args:
        trajectory (list[numpy.ndarray]): список состояний траектории

    Returns:
        numpy.ndarray: точка падения [x, y, vx, vy] при y = 0
    """
    if len(trajectory) < 2:
        return trajectory[-1]

    # Находим последние две точки траектории
    state_prev = trajectory[-2]
    state_curr = trajectory[-1]

    # Линейная интерполяция для точного определения x при y = 0
    if state_curr[1] < 0 and state_prev[1] >= 0:
        # Пропорция времени, когда y пересекает 0
        ratio = -state_prev[1] / (state_curr[1] - state_prev[1])

        # Интерполируем все компоненты состояния
        interpolated_state = state_prev + ratio * (state_curr - state_prev)
        interpolated_state[1] = 0.0  # Точно устанавливаем y = 0
        return interpolated_state

    return state_curr

In [ ]:
def integrate_trajectory(theta, return_full_trajectory=False):
    """
    Интегрирование полной траектории снаряда для заданного угла стрельбы.

    Функция моделирует полет снаряда от начальной точки до падения на землю,
    учитывая гравитацию, сопротивление воздуха и горизонтальный ветер.

    Args:
        theta (float): начальный угол стрельбы относительно горизонта, радианы
        return_full_trajectory (bool): если True, возвращает массив всех состояний траектории

    Returns:
        tuple или numpy.ndarray:
            - Если return_full_trajectory=False: финальное состояние [x, y, vx, vy] при падении
            - Если return_full_trajectory=True: (финальное состояние, траектория)
              где траектория - numpy.ndarray формы (n_points, 4) с состояниями [x, y, vx, vy]
    """
    # Инициализация начального состояния
    initial_state = initialize_state(theta)

    # Интегрирование до падения на землю
    final_state, trajectory = integrate_until_ground(initial_state)

    if return_full_trajectory:
        # Интерполяция для точного определения точки падения
        trajectory_array = np.array(trajectory)
        interpolated_final = interpolate_ground_impact(trajectory)
        return interpolated_final, trajectory_array
    else:
        return final_state

## 3. Метод стрельбы ##

In [ ]:
def residual_function(theta):
    """
    Вычисление функции невязки для метода стрельбы.

    Невязка показывает отклонение точки падения снаряда от целевой координаты x.

    Args:
        theta (float): начальный угол стрельбы, радианы

    Returns:
        float: невязка (x_final - x_target), метры
               положительная - перелет, отрицательная - недолет
    """
    final_state = integrate_trajectory(theta)
    x_final = final_state[0]

    return x_final - x_target

In [ ]:
def initialize_shooting_bounds(theta_left, theta_right):
    """
    Инициализация границ поиска для метода стрельбы.

    Args:
        theta_left (float): левая граница угла стрельбы, радианы
        theta_right (float): правая граница угла стрельбы, радианы

    Returns:
        tuple: (residual_left, residual_right, theta_history, residual_history)
               где theta_history и residual_history содержат начальные значения
    """
    residual_left = residual_function(theta_left)
    residual_right = residual_function(theta_right)

    theta_history = [theta_left, theta_right]
    residual_history = [residual_left, residual_right]

    return residual_left, residual_right, theta_history, residual_history

In [ ]:
def check_convergence(residual_left, residual_right, tolerance):
    """
    Проверка условий сходимости метода стрельбы.

    Args:
        residual_left (float): невязка левой границы
        residual_right (float): невязка правой границы
        tolerance (float): требуемая точность решения

    Returns:
        tuple: (converged, theta_solution)
               converged - True если решение найдено
               theta_solution - найденный угол или None
    """
    if abs(residual_left) < tolerance:
        return True, None  # Решение в левой границе
    if abs(residual_right) < tolerance:
        return True, None  # Решение в правой границе
    if abs(residual_right - residual_left) < tolerance:
        raise ValueError("Метод не сошелся: невязки слишком близки")

    return False, None

In [ ]:
def compute_new_theta_regula_falsi(theta_left, theta_right, residual_left, residual_right):
    """
    Вычисление нового угла методом regula falsi (ложной позиции).

    Args:
        theta_left (float): левая граница угла, радианы
        theta_right (float): правая граница угла, радианы
        residual_left (float): невязка левой границы
        residual_right (float): невязка правой границы

    Returns:
        float: новый угол стрельбы, радианы
    """
    return (residual_right * theta_left - residual_left * theta_right) / (residual_right - residual_left)

In [ ]:
def update_bounds_regula_falsi(theta_left, theta_right, residual_left, residual_right,
                               theta_new, residual_new):
    """
    Обновление границ поиска методом regula falsi.

    Выбирает новую пару точек в зависимости от знаков невязок для обеспечения сходимости.

    Args:
        theta_left (float): текущая левая граница, радианы
        theta_right (float): текущая правая граница, радианы
        residual_left (float): невязка левой границы
        residual_right (float): невязка правой границы
        theta_new (float): новый кандидат угла, радианы
        residual_new (float): невязка нового кандидата

    Returns:
        tuple: (новая_левая_граница, новая_правая_граница,
                новая_невязка_левой, новая_невязка_правой)
    """
    if residual_right * residual_left > 0:
        # Невязки одного знака - выбираем точку ближе к новой
        if abs(theta_new - theta_left) > abs(theta_new - theta_right):
            return theta_new, theta_right, residual_new, residual_right
        else:
            return theta_left, theta_new, residual_left, residual_new
    else:
        # Невязки разных знаков - выбираем точку с противоположным знаком
        if residual_new * residual_left > 0:
            return theta_new, theta_right, residual_new, residual_right
        else:
            return theta_left, theta_new, residual_left, residual_new

In [ ]:
def shooting_method(theta_left, theta_right, max_iterations=50):
    """
    Решение обратной баллистической задачи методом стрельбы с использованием regula falsi.

    Метод итеративно уточняет угол стрельбы, чтобы снаряд попадал точно в цель,
    используя комбинацию методов бисекции и секущих для быстрой сходимости.

    Args:
        theta_left (float): левая граница поиска угла стрельбы, радианы
        theta_right (float): правая граница поиска угла стрельбы, радианы
        max_iterations (int): максимальное число итераций поиска

    Returns:
        tuple: (theta_solution, theta_history, residual_history)
               theta_solution (float): найденный угол стрельбы, радианы
               theta_history (list[float]): история углов на всех итерациях
               residual_history (list[float]): история невязок на всех итерациях

    Raises:
        ValueError: если метод не сошелся за заданное число итераций
    """
    # Инициализация границ и истории
    residual_left, residual_right, theta_history, residual_history = initialize_shooting_bounds(
        theta_left, theta_right)

    print("Начало метода стрельбы:")
    print(f"θ_left = {np.degrees(theta_left):.4f}°, residual_left = {residual_left:.4f} м")
    print(f"θ_right = {np.degrees(theta_right):.4f}°, residual_right = {residual_right:.4f} м")

    # Основной цикл итераций
    for iteration in range(max_iterations):
        # Проверка условий сходимости
        converged, theta_solution = check_convergence(residual_left, residual_right, tolerance)

        if converged:
            if theta_solution is None:  # Решение найдено в одной из границ
                theta_solution = theta_left if abs(residual_left) < tolerance else theta_right
            print(f"Решение найдено: θ = {np.degrees(theta_solution):.4f}°")
            return theta_solution, theta_history, residual_history

        # Вычисление нового кандидата угла
        theta_new = compute_new_theta_regula_falsi(theta_left, theta_right, residual_left, residual_right)
        residual_new = residual_function(theta_new)

        # Сохранение в истории
        theta_history.append(theta_new)
        residual_history.append(residual_new)

        print(f"Итерация {iteration+1}: θ = {np.degrees(theta_new):.4f}°, невязка = {residual_new:.4f} м")

        # Обновление границ поиска
        theta_left, theta_right, residual_left, residual_right = update_bounds_regula_falsi(
            theta_left, theta_right, residual_left, residual_right, theta_new, residual_new)

    raise ValueError(f"Метод не сошелся за {max_iterations} итераций")

## 4. Демонстрация метода стрельбы ##

In [ ]:
# Настройка границ поиска
theta_left = np.radians(10)   # 10 градусов - снаряд не долетит
theta_right = np.radians(80)  # 80 градусов - снаряд перелетит

# Вывод заголовка и параметров задачи
print("="*60)
print("МЕТОД СТРЕЛЬБЫ: ОБРАТНАЯ БАЛЛИСТИЧЕСКАЯ ЗАДАЧА")
print("="*60)
print(f"Целевая точка: x = {x_target} м, y = {y_target} м")
print(f"Начальная скорость: v0 = {v0} м/с")
print(f"Сопротивление воздуха: k = {k} 1/с")
print(f"Горизонтальный ветер: wx = {wind_x} м/с")
print()

# Запуск метода стрельбы
theta_shooting, theta_hist_shooting, residual_hist_shooting = shooting_method(theta_left, theta_right)

# Вывод результатов
print("\nРезультат метода стрельбы:")
print(f"Угол стрельбы: {np.degrees(theta_shooting):.6f}°")
print(f"Невязка: {residual_hist_shooting[-1]:.2e}")

# Проверка решения
final_state = integrate_trajectory(theta_shooting)
print(f"Достигнутая точка: x = {final_state[0]:.3f} м, y = {final_state[1]:.3f} м")

# Визуализация первых траекторий
print("\nВизуализация первых траекторий метода стрельбы...")
n_trajectories = min(8, len(theta_hist_shooting))
plot_trajectories(theta_hist_shooting[:n_trajectories], integrate_trajectory,
                 x_target, y_target, x_start, y_start,
                 f"Первые {n_trajectories} траекторий метода стрельбы (ветер: {wind_x} м/с)")

# Визуализация процесса сходимости
plot_convergence(theta_hist_shooting, residual_hist_shooting, "Метод стрельбы")

## Выводы ##

## Результаты решения обратной баллистической задачи методом стрельбы

Метод стрельбы успешно решил обратную баллистическую задачу, найдя угол стрельбы,
при котором снаряд попадает точно в цель с учетом сопротивления воздуха и ветра.

### Ключевые особенности решения:

1. **Начальные границы**: Были выбраны углы 10° и 80°, при которых снаряд
   не долетает до цели, но метод все равно сошелся благодаря свойствам regula falsi

2. **Процесс сходимости**: Метод показал постепенное приближение к правильному углу
   через последовательные "выстрелы" с корректировкой угла

3. **Визуализация**: Графики показывают, как каждая итерация приближает траекторию
   к целевой точке, демонстрируя принцип "пристрелки"

4. **Точность**: Метод достиг высокой точности (невязка ~1e-10) за небольшое
   число итераций